<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_08_model_tuning/stage_08_02_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_08_02 - TUNING - XGBOOST**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [28]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-12 23:53:50,648 | INFO | Environment initialized


## **2. Acceso a drive**

In [29]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-12 23:53:53,092 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [30]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
    "t2_dir_thr_90",
    "t2_dir_thr_120",
]

# Tamaños de ventana
WINDOW_SIZES = [30, 60, 90, 120, 180]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
    "regime_id",
    "roc_30",
    "roc_60",
    "stoch_k_30",
    "atr_norm_10",
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-12 23:53:53,099 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-12 23:53:53,099 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-12 23:53:53,100 | INFO | Configuración de experimento cargada
2026-04-12 23:53:53,100 | INFO | Targets: ['t2_dir_thr_90', 't2_dir_thr_120']
2026-04-12 23:53:53,101 | INFO | Window sizes: [30, 60, 90, 120, 180]


In [31]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:5]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[60]["t2_dir_thr_90"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-12 23:53:53,110 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-12 23:53:53,117 | INFO | Windows OK      : 30
2026-04-12 23:53:53,118 | INFO | Windows missing : 0
2026-04-12 23:53:53,118 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_t2.pkl
2026-04-12 23:53:53,119 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [32]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [33]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [34]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_dir_thr_90'
        - 't2_dir_thr_120'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }


### **4.4. Creación de bundles T2**

In [35]:
# --------------------------------------------------
# Crea bundles T2 para un window_size dado
# --------------------------------------------------
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundle_t2_90  : dict
    bundle_t2_120 : dict
    """

    if len(targets) != 2:
        raise ValueError(
            f"Se esperaban exactamente 2 targets T2. Recibido: {targets}"
        )

    bundle_t2_90 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[0],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    bundle_t2_120 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[1],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    # --------------------------------------------------
    # Verificación rápida
    # --------------------------------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    print(f"\nTARGET: {targets[0]}")
    print("Train :", bundle_t2_90["train"]["X"].shape, bundle_t2_90["train"]["y"].shape)
    print("Valid :", bundle_t2_90["valid"]["X"].shape, bundle_t2_90["valid"]["y"].shape)
    print("Test  :", bundle_t2_90["test"]["X"].shape,  bundle_t2_90["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_90["scaler"]).__name__)

    print(f"\nTARGET: {targets[1]}")
    print("Train :", bundle_t2_120["train"]["X"].shape, bundle_t2_120["train"]["y"].shape)
    print("Valid :", bundle_t2_120["valid"]["X"].shape, bundle_t2_120["valid"]["y"].shape)
    print("Test  :", bundle_t2_120["test"]["X"].shape,  bundle_t2_120["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_120["scaler"]).__name__)

    return bundle_t2_90, bundle_t2_120

In [36]:
#bundle_t2_90, bundle_t2_120 = create_bundles(window_size=60)

Como acceder a las ventanas X e y:

```python
X_train_90 = bundle_t2_90["train"]["X"]
y_train_90 = bundle_t2_90["train"]["y"]

X_valid_120 = bundle_t2_120["valid"]["X"]
y_valid_120 = bundle_t2_120["valid"]["y"]

scaler = bundle_t2_90["scaler"]
```




### **4.5. Preparación de inputs según el tipo de modelo**

In [37]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

In [38]:
from __future__ import annotations

from typing import Any, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)

    Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1). Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )

In [39]:
# ============================================================
# 2) Sanity check principal (seq2one clasificación T2)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como d_flat esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado).
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len), permite tomar y[:, -1].
        Por defecto False.
    """

    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Normalización de y
    # --------------------------------------------------
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        y = y[:, -1]

    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Inferir modo y dimensiones de X
    # --------------------------------------------------
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # --------------------------------------------------
    # Validación básica de n_samples
    # --------------------------------------------------
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # --------------------------------------------------
    # Validación de shapes según modo
    # --------------------------------------------------
    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    # --------------------------------------------------
    # Diagnóstico de clases
    # --------------------------------------------------
    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    # --------------------------------------------------
    # Salida informativa
    # --------------------------------------------------
    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info

In [40]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_dir_thr_90",
      "horizon": 90,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # --------------------------------------------------
    # Setear esperados desde TRAIN si no se dieron
    # --------------------------------------------------
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    # --------------------------------------------------
    # Ejecutar checks
    # --------------------------------------------------
    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_targets_seq2one(
    bundle_t2_90: Dict[str, Any],
    bundle_t2_120: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para ambos targets T2.
    """
    out_t2_90 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_90,
        tag="t2_90",
        verbose=verbose,
    )

    out_t2_120 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_120,
        tag="t2_120",
        verbose=verbose,
    )

    return {
        "t2_dir_thr_90": out_t2_90,
        "t2_dir_thr_120": out_t2_120,
    }

In [41]:
#sanity_outputs = run_sanity_checks_all_targets_seq2one(
#    bundle_t2_90,
#    bundle_t2_120,
#    verbose=True,
#)

In [42]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **6. Métricas de clasificación T2**

In [43]:
# ================================
# Setup para importar módulos del proyecto
# ================================

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas (T2 clasificación)
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

# ================================
# Utilidad: métricas → DataFrame
# ================================

import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte métricas de clasificación T2 en una fila de DataFrame.
    """

    m = metrics["metrics"]

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "n_samples": m["n_samples"],

        # principales
        "balanced_accuracy": m["balanced_accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],

        # complementarias
        "accuracy": m["accuracy"],
        "precision_macro": m["precision_macro"],
        "recall_macro": m["recall_macro"],

        # baseline
        "balanced_accuracy_naive": m.get("balanced_accuracy_naive"),
        "bal_acc_gain_vs_naive": m.get("balanced_accuracy_gain_vs_naive"),
    }])

logger.info("Métricas T2 + utilidades cargadas")

2026-04-12 23:53:53,204 | INFO | Métricas T2 + utilidades cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [44]:
# ================================
# Carga de métricas (si existen)
# ================================

def load_classification_metrics_if_exists(
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/tuning_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.
    """

    path = base_dir / f"classification_{name}_metrics.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(f"No existen métricas previas para: {name}")
    return pd.DataFrame()

In [45]:
# ================================
# Guardado de métricas
# ================================

def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/tuning_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """

    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_{name}_metrics.parquet"

    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path

Ejemplo de uso:

```python
df_metrics = load_classification_metrics_if_exists(name="lstm_valid")

df_metrics = pd.concat([df_metrics, new_row_df], ignore_index=True)

save_classification_metrics(df_metrics, name="lstm_valid")
```



## **8. Gestión de dispositivo y memoria**

In [46]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [47]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-12 23:53:53,229 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo XGBoost**

**Tuning grueso (primero)**

👉 Ajustamos solo lo más importante:

* `n_estimators`
* `max_depth`
* `learning_rate`
* `subsample`
* `colsample_bytree`

**- Espacio de búsqueda (tuning grueso)**

```python
xgb_coarse_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}
```
**- Interpretación rápida**

* `n_estimators` → pocos vs muchos árboles
* `max_depth` → modelo simple vs complejo
* `learning_rate` → conservador vs agresivo
* `subsample`, `colsample_bytree` → regularización básica

**- Tamaño del grid**

 3 × 3 × 3 × 2 × 2 = **108 combinaciones por escenario**

Y tú tienes:

 6 escenarios → **648 runs**


---

**Tuning fino (después)**

👉 Ajustamos regularización y detalles:

* `min_child_weight`
* `gamma`
* `reg_alpha`
* `reg_lambda`

---

**Resumen claro**

* **Grueso** → estructura del modelo
* **Fino** → regularización y ajuste fino


## **10.1. Función unitaria por bundle**

In [48]:
from xgboost import XGBClassifier
import numpy as np


def run_xgboost_for_bundle_seq2one_coarse(
    bundle,
    *,
    # hiperparámetros del grid
    n_estimators,
    max_depth,
    learning_rate,
    subsample,
    colsample_bytree,

    # fijos en tuning grueso
    reg_alpha=0.0,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    input_mode="2d_flat",
    class_weight=None,
    verbose=False,
):
    """
    XGBoost para tuning grueso.

    - TRAIN → fit
    - VALID → evaluación
    - NO usa TEST
    """

    # =========================
    # 1. DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    # =========================
    # 2. INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)

    # =========================
    # 3. ENCODE LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))
    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    y_valid_enc = np.array([class_to_idx[y] for y in y_valid], dtype=np.int32)

    num_class = len(classes_)

    # =========================
    # 4. SAMPLE WEIGHT
    # =========================
    sample_weight = None

    if class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_class * count)
            for idx, count in enumerate(counts)
        }
        sample_weight = np.array(
            [weights_by_idx[idx] for idx in y_train_enc],
            dtype=np.float32,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }
        sample_weight = np.array(
            [weights_by_idx.get(idx, 1.0) for idx in y_train_enc],
            dtype=np.float32,
        )

    # =========================
    # 5. MODELO
    # =========================
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        tree_method="hist",
        device="cuda",  # ← GPU
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric="mlogloss",
        verbosity=1 if verbose else 0,
    )

    # =========================
    # 6. TRAIN
    # =========================
    model.fit(X_train_model, y_train_enc, sample_weight=sample_weight)

    # =========================
    # 7. PREDICT (VALID ONLY)
    # =========================
    y_pred_valid_enc = model.predict(X_valid_model)
    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])

    y_proba_valid = model.predict_proba(X_valid_model)

    return {
        "model": model,
        "y_true_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
        "params": {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
        }
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [49]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_xgboost_bundles_coarse(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    split: str = "valid",
    model_name: str = "xgboost",

    # hiperparámetros (tuning grueso)
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,

    # fijos
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,

    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Evalúa XGBoost para tuning grueso (SOLO VALID).

    - TRAIN → fit
    - VALID → evaluación
    - NO usa TEST
    """

    # --------------------------------------------------
    # 1) Normalizar entrada
    # --------------------------------------------------
    if isinstance(bundles, dict):
        bundles_list: List[Dict[str, Any]] = [bundles]
    else:
        bundles_list = list(bundles)

    # --------------------------------------------------
    # 2) Validar split
    # --------------------------------------------------
    if split != "valid":
        raise ValueError("Para tuning, split debe ser únicamente 'valid'")

    rows = []

    # --------------------------------------------------
    # 3) Iterar bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"model={model_name} | "
                f"n_est={n_estimators} | "
                f"depth={max_depth} | "
                f"lr={learning_rate} | "
                f"sub={subsample} | "
                f"col={colsample_bytree}"
            )

        # ----------------------------------------------
        # 4) Entrenar + predecir SOLO VALID
        # ----------------------------------------------
        preds = run_xgboost_for_bundle_seq2one_coarse(
            bundle,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            verbose=False,
        )

        # ----------------------------------------------
        # 5) y_true / y_pred VALID
        # ----------------------------------------------
        y_true = preds["y_true_valid"]
        y_pred = preds["y_pred_valid"]

        # ----------------------------------------------
        # 6) Métricas
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split="valid",
            target=target,
            labels=[-1, 0, 1],
        )

        # ----------------------------------------------
        # 7) A DataFrame
        # ----------------------------------------------
        df_row = metrics_to_df(
            metrics,
            model=model_name,
            split="valid",
            window_size=window_size,
            target=target,
        )

        # metadatos
        df_row["horizon_min"] = horizon
        df_row["class_weight_mode"] = class_weight

        # hiperparámetros
        df_row["n_estimators"] = n_estimators
        df_row["max_depth"] = max_depth
        df_row["learning_rate"] = learning_rate
        df_row["subsample"] = subsample
        df_row["colsample_bytree"] = colsample_bytree

        rows.append(df_row)

    # --------------------------------------------------
    # 8) Consolidar
    # --------------------------------------------------
    return pd.concat(rows, ignore_index=True)

## **10.3. Función orquestadora por `window_size`**

In [50]:
import gc
import pandas as pd


def run_xgboost(
    window_size: int,
    *,
    verbose: bool = True,
    model_name: str = "xgboost",

    # hiperparámetros tuning grueso
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,

    # fijos
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,

    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight: str = "balanced",
) -> pd.DataFrame:
    """
    Ejecuta XGBoost para una sola window_size usando SOLO VALID.

    - Entrena en TRAIN
    - Evalúa SOLO en VALID
    - Pensado para tuning grueso
    """

    size = int(window_size)

    bundle_t2_90 = bundle_t2_120 = None
    bundles_t2 = None
    df_out = None

    model_name_effective = f"{model_name}_balanced"

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"XGBOOST | TUNING | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"n_estimators     = {n_estimators}")
            print(f"max_depth        = {max_depth}")
            print(f"learning_rate    = {learning_rate}")
            print(f"subsample        = {subsample}")
            print(f"colsample_bytree = {colsample_bytree}")

        # --------------------------------------------------
        # 2) Construcción de bundles
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size}")

        bundle_t2_90, bundle_t2_120 = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        bundles_t2 = [bundle_t2_90, bundle_t2_120]

        # --------------------------------------------------
        # 3) Evaluación SOLO VALID
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | model={model_name_effective}"
            )

        df_out = eval_xgboost_bundles_coarse(
            bundles_t2,
            split="valid",
            model_name=model_name_effective,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            verbose=verbose,
        )

        # --------------------------------------------------
        # 4) Orden final
        # --------------------------------------------------
        df_out = (
            df_out
            .sort_values(["window_size", "target", "horizon_min"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "target",
                        "model",
                        "n_estimators",
                        "max_depth",
                        "learning_rate",
                        "subsample",
                        "colsample_bytree",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ]
                .to_string(index=False)
            )

        return df_out

    finally:
        # --------------------------------------------------
        # 6) Liberación de memoria
        # --------------------------------------------------
        del bundle_t2_90, bundle_t2_120, bundles_t2
        gc.collect()

## **10.4. Función incremental multi-ventana**

In [51]:
from pathlib import Path
import gc
import pandas as pd


def run_xgboost_incremental(
    *,
    window_sizes: list[int],
    name: str = "xgboost",
    verbose: bool = True,

    # hiperparámetros tuning grueso
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,

    # fijos
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,

    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
) -> pd.DataFrame:
    """
    Ejecuta XGBoost incremental para tuning (SOLO VALID).

    - una combinación fija de hiperparámetros por corrida
    - hace skip si esa combinación ya fue corrida
    - usa SOLO split='valid'
    """

    class_weight = "balanced"
    name_effective = f"{name}_balanced"

    metrics_dir = DRIVE_DIR / "metrics/tuning_metrics"
    metrics_path = metrics_dir / f"classification_{name_effective}_metrics.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Esperado por corrida
    # --------------------------------------------------
    expected_combos = {
        ("t2_dir_thr_90", "valid"),
        ("t2_dir_thr_120", "valid"),
    }

    # --------------------------------------------------
    # 3) Loop por window_sizes
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)

        df_existing = None

        # ----------------------------------------------
        # 3.1) Filtrar histórico (misma config exacta)
        # ----------------------------------------------
        if not df_hist.empty:
            mask = (
                (df_hist["window_size"] == L)
                & (df_hist["model"] == name_effective)
                & (df_hist["class_weight_mode"] == "balanced")
                & (df_hist["n_estimators"] == n_estimators)
                & (df_hist["max_depth"] == max_depth)
                & (df_hist["learning_rate"] == learning_rate)
                & (df_hist["subsample"] == subsample)
                & (df_hist["colsample_bytree"] == colsample_bytree)
            )

            df_existing = df_hist.loc[mask]

            if not df_existing.empty:
                combos_done = set(zip(df_existing["target"], df_existing["split"]))
                is_complete = expected_combos.issubset(combos_done)
            else:
                is_complete = False

            if is_complete:
                if verbose:
                    print(f"[SKIP] {name_effective} | L={L} | HP ya existe")
                continue

        # ----------------------------------------------
        # 3.2) Ejecutar
        # ----------------------------------------------
        if verbose:
            print("\n" + "-" * 100)
            print(f"[RUN] {name_effective} | L={L}")
            print(
                f"n_estimators={n_estimators} | "
                f"max_depth={max_depth} | "
                f"learning_rate={learning_rate} | "
                f"subsample={subsample} | "
                f"colsample_bytree={colsample_bytree}"
            )
            print("-" * 100)

        df_L = run_xgboost(
            window_size=L,
            verbose=verbose,
            model_name=name,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight="balanced",
        )

        # familia
        df_L["family"] = name_effective

        # ----------------------------------------------
        # 3.3) Actualizar histórico
        # ----------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ----------------------------------------------
        # 3.4) Drop duplicates (clave tuning)
        # ----------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "horizon_min",
            "class_weight_mode",
            "n_estimators",
            "max_depth",
            "learning_rate",
            "subsample",
            "colsample_bytree",
        ]

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # 3.5) Guardar
        # ----------------------------------------------
        save_classification_metrics(df_hist, name=name_effective)

        # ----------------------------------------------
        # 3.6) Liberar memoria
        # ----------------------------------------------
        del df_L, df_existing
        gc.collect()

    # --------------------------------------------------
    # 4) Orden final
    # --------------------------------------------------
    sort_cols = [
        "window_size",
        "target",
        "split",
        "horizon_min",
        "model",
        "n_estimators",
        "max_depth",
        "learning_rate",
    ]

    return df_hist.sort_values(sort_cols).reset_index(drop=True)

## **10.5. Ejecución final del experimento**

In [52]:
from itertools import product
from joblib import Parallel, delayed
import pandas as pd

# ============================================================
# 1) Grid de hiperparámetros
# ============================================================

param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

window_sizes = [30, 60, 180]

name = "xgboost"
random_state = 42
n_jobs = -1
input_mode = "2d_flat"

# ============================================================
# 2) Preparar tareas
# ============================================================

keys, values = zip(*param_grid.items())
all_combos = list(product(*values))

tasks = []
for combo_idx, combo in enumerate(all_combos, start=1):
    params = dict(zip(keys, combo))
    for L in window_sizes:
        tasks.append((combo_idx, params, L))

print("\n" + "=" * 100)
print(f"TOTAL TASKS: {len(tasks)}")
print("=" * 100)

# ============================================================
# 3) Función por tarea
# ============================================================

def run_task(task):
    combo_idx, params, L = task

    print(
        f"[RUN] combo={combo_idx} | L={L} | "
        f"n_estimators={params['n_estimators']} | "
        f"max_depth={params['max_depth']} | "
        f"learning_rate={params['learning_rate']} | "
        f"subsample={params['subsample']} | "
        f"colsample_bytree={params['colsample_bytree']}"
    )

    df_L = run_xgboost(
        window_size=L,
        verbose=False,
        model_name=name,
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=random_state,
        n_jobs=n_jobs,
        input_mode=input_mode,
        class_weight="balanced",
    )

    df_L["family"] = f"{name}_balanced"
    return df_L

# ============================================================
# 4) Ejecutar en paralelo
# ============================================================

results = Parallel(n_jobs=3, verbose=10)(
    delayed(run_task)(task) for task in tasks
)

# ============================================================
# 5) Consolidar y guardar una sola vez
# ============================================================

df_hist = pd.concat(results, ignore_index=True)

subset_cols = [
    "window_size",
    "target",
    "split",
    "model",
    "horizon_min",
    "class_weight_mode",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
]

df_hist = (
    df_hist
    .drop_duplicates(subset=subset_cols, keep="last")
    .sort_values([
        "window_size",
        "target",
        "split",
        "horizon_min",
        "model",
        "n_estimators",
        "max_depth",
        "learning_rate",
    ])
    .reset_index(drop=True)
)

save_classification_metrics(df_hist, name="xgboost_balanced")

print("\n" + "=" * 100)
print("[DONE] Tuning grueso XGBoost finalizado")
print("=" * 100)


TOTAL TASKS: 324


[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done   2 tasks      | elapsed:   25.5s
[Parallel(n_jobs=3)]: Done   7 tasks      | elapsed:  1.3min
[Parallel(n_jobs=3)]: Done  12 tasks      | elapsed:  2.1min
[Parallel(n_jobs=3)]: Done  19 tasks      | elapsed:  3.3min
[Parallel(n_jobs=3)]: Done  26 tasks      | elapsed:  4.5min
[Parallel(n_jobs=3)]: Done  35 tasks      | elapsed:  6.0min
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:  7.8min
[Parallel(n_jobs=3)]: Done  55 tasks      | elapsed: 10.1min
[Parallel(n_jobs=3)]: Done  66 tasks      | elapsed: 12.1min
[Parallel(n_jobs=3)]: Done  79 tasks      | elapsed: 15.2min
[Parallel(n_jobs=3)]: Done  92 tasks      | elapsed: 18.5min
[Parallel(n_jobs=3)]: Done 107 tasks      | elapsed: 22.2min
[Parallel(n_jobs=3)]: Done 122 tasks      | elapsed: 26.1min
[Parallel(n_jobs=3)]: Done 139 tasks      | elapsed: 30.4min
[Parallel(n_jobs=3)]: Done 156 tasks      | elapsed: 36.0min
[Parallel(


[DONE] Tuning grueso XGBoost finalizado


In [53]:
df_hist

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,...,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,family
0,xgboost_balanced,valid,30,t2_dir_thr_120,93508,0.419975,0.424470,0.646521,0.662168,0.434086,...,0.333333,0.086641,120,balanced,100,3,0.03,0.8,0.8,xgboost_balanced
1,xgboost_balanced,valid,30,t2_dir_thr_120,93508,0.421113,0.425870,0.647206,0.663034,0.435807,...,0.333333,0.087780,120,balanced,100,3,0.03,0.8,1.0,xgboost_balanced
2,xgboost_balanced,valid,30,t2_dir_thr_120,93508,0.421043,0.425429,0.645106,0.658778,0.433437,...,0.333333,0.087709,120,balanced,100,3,0.03,1.0,0.8,xgboost_balanced
3,xgboost_balanced,valid,30,t2_dir_thr_120,93508,0.421658,0.425916,0.644456,0.657484,0.433500,...,0.333333,0.088324,120,balanced,100,3,0.03,1.0,1.0,xgboost_balanced
4,xgboost_balanced,valid,30,t2_dir_thr_120,93508,0.419292,0.423523,0.643702,0.656842,0.430877,...,0.333333,0.085958,120,balanced,100,3,0.10,0.8,0.8,xgboost_balanced
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
643,xgboost_balanced,valid,180,t2_dir_thr_90,64408,0.382852,0.380169,0.574790,0.623463,0.412389,...,0.333333,0.049518,90,balanced,500,7,0.10,1.0,1.0,xgboost_balanced
644,xgboost_balanced,valid,180,t2_dir_thr_90,64408,0.370724,0.360928,0.567303,0.631785,0.408386,...,0.333333,0.037391,90,balanced,500,7,0.20,0.8,0.8,xgboost_balanced
645,xgboost_balanced,valid,180,t2_dir_thr_90,64408,0.369449,0.358975,0.566559,0.631863,0.406443,...,0.333333,0.036116,90,balanced,500,7,0.20,0.8,1.0,xgboost_balanced
646,xgboost_balanced,valid,180,t2_dir_thr_90,64408,0.369260,0.358670,0.567127,0.632375,0.404984,...,0.333333,0.035927,90,balanced,500,7,0.20,1.0,0.8,xgboost_balanced
